# From hidden supplier risk to a mitigation plan

## The situation

One day, a tier-3 supplier **`S-T3-07`** announces it can't deliver. We don't buy from it directly: it supplies our suppliers' suppliers. Somewhere downstream, customer orders depend on it, but nobody can say which ones.

## The question this notebook answers

> **Which customer orders are at risk because `S-T3-07` stopped delivering, why, and what should we do about it?**

By the end, we have three answers:

1. **Which:** the at-risk orders, their value, and their regions
2. **Why:** the chain of supplier dependencies from `S-T3-07` to each affected part
3. **What to do:** a mitigation plan, written by an LLM from checked facts

Answering this means following **two networks of unknown depth** and joining them:

1. **Up the supply chain:** which suppliers rely on `S-T3-07`, at any tier? The network has a cycle (`S-T2-11` ↔ `S-T2-14`).
2. **Down the bill of materials:** which products contain a part that has lost all of its suppliers, at any level?

Then we hand the result to an LLM to write a mitigation plan. Each tool does the part it is good at:

| Step | Tool | Why this tool |
|---|---|---|
| Store the data, simple aggregates | Snowflake SQL | Tables, joins, sums |
| Multi-hop reasoning over both networks | RelationalAI (PyRel) | Recursive rules, cycles handled, results stay reusable in the model |
| Turn facts into a readable plan | Cortex `AI_COMPLETE` | Language, structure, trade-offs |
| Check the plan against the facts | Python | Trust, but verify |


> **The LLM never sees the raw tables. It gets a small facts pack that RelationalAI derived. It writes; it does not calculate.**

## Before you start

- Run notebook `01_relationalai_setup.ipynb` first. It loads the tables and checks your connection.
- Select the kernel **RAI supply chain**. PyRel finds `raiconfig.yaml` in the guide folder, one level up, and the notebook imports `supply_chain_ontology.py` from this folder.
- `RAI_DEVELOPER` needs Cortex access for `AI_COMPLETE` (the `SNOWFLAKE.CORTEX_USER` database role, granted to `PUBLIC` by default).

**Check notebook 01 completed succesfully:**

In [ ]:
import datetime as dt
import importlib.metadata as md
import json
import re

import pandas as pd
from supply_chain_ontology import model

session = model.config.get_session()   # the same connection PyRel uses, from raiconfig.yaml
session.sql("USE SCHEMA RAI_QUICKSTART.SUPPLY_CHAIN").collect()
print("relationalai", md.version("relationalai"))


def sql(query, params=None):
    """Run SQL and return a pandas DataFrame with lower-case column names."""
    df = session.sql(query, params=params).to_pandas()
    df.columns = [c.lower() for c in df.columns]
    return df


FAILED = "S-T3-07"                 # the supplier that stops delivering
AS_OF = dt.date(2026, 10, 26)      # the day we hear about it
LLM = "claude-sonnet-4-5"          # any AI_COMPLETE model available in your region

## 1. What the tier-1 view shows

The sensor `P-141` has two suppliers. That looks safe: if one fails, the other takes over.

In [ ]:
sql("""
SELECT sp.part_id, p.part_name, sp.supplier_id, s.tier, s.region, sp.capacity_per_week, sp.unit_price
FROM SUPPLIER_PARTS sp JOIN SUPPLIERS s USING (supplier_id) JOIN PARTS p USING (part_id)
WHERE sp.part_id = 'P-141'
""")

## 2. The SQL attempt: possible, but with complications

Recursive CTEs can follow both networks. Two things make it fragile.

**The cycle.** `S-T2-11` and `S-T2-14` depend on each other. A recursive CTE that walks the supplier network without a guard goes round that loop forever. `load_data.sql` sets a 600-second statement timeout on `RAI_QS_WH`. **Don't run this one:**

```sql
-- "who relies on whom, at any tier": never ends on a cyclic network
WITH RECURSIVE relies_on (buyer, upstream) AS (
    SELECT supplier_id, depends_on_supplier_id FROM SUPPLIER_DEPENDENCIES
    UNION ALL
    SELECT r.buyer, d.depends_on_supplier_id
    FROM relies_on r JOIN SUPPLIER_DEPENDENCIES d ON d.supplier_id = r.upstream
)
SELECT DISTINCT * FROM relies_on;
```

Whether a query hits the loop depends on where it starts. Starting from `S-T3-07` happens to avoid it today, but one new dependency in the data can change that. You find out when it doesn't finish.

**The guard is your job.** You carry the visited path in an array, skip suppliers already on it, and cap the depth. Then you need a second recursion for the bill of materials, and set logic for "every supplier of this part is down".

Here is the guarded version. It works, and it is correct for this data:

In [ ]:
# SQL, with the guard: carry the visited path, skip suppliers already on it, cap the depth.
impact_sql = sql("""
WITH RECURSIVE down_suppliers (supplier_id, path, depth) AS (
    SELECT supplier_id, ARRAY_CONSTRUCT(supplier_id), 0 FROM SUPPLIERS WHERE supplier_id = ?
    UNION ALL
    SELECT d.supplier_id, ARRAY_APPEND(u.path, d.supplier_id), u.depth + 1
    FROM down_suppliers u JOIN SUPPLIER_DEPENDENCIES d ON d.depends_on_supplier_id = u.supplier_id
    WHERE NOT ARRAY_CONTAINS(d.supplier_id::VARIANT, u.path) AND u.depth < 10
),
down AS (SELECT DISTINCT supplier_id FROM down_suppliers),
unsourced AS (                       -- parts whose every supplier is down
    SELECT part_id FROM SUPPLIER_PARTS
    GROUP BY part_id HAVING COUNT_IF(supplier_id NOT IN (SELECT supplier_id FROM down)) = 0
),
explode (product_id, part_id) AS (   -- second recursion: the bill of materials
    SELECT p.product_id, b.child_part_id FROM PRODUCTS p JOIN BOM b ON b.parent_part_id = p.part_id
    UNION ALL
    SELECT e.product_id, b.child_part_id FROM explode e JOIN BOM b ON b.parent_part_id = e.part_id
),
hit AS (SELECT DISTINCT product_id FROM explode WHERE part_id IN (SELECT part_id FROM unsourced))
SELECT (SELECT ARRAY_AGG(supplier_id) WITHIN GROUP (ORDER BY supplier_id) FROM down) AS down_suppliers,
       (SELECT ARRAY_AGG(part_id) FROM unsourced) AS parts_without_supplier,
       COUNT(*) AS affected_orders, SUM(order_value) AS affected_value
FROM ORDERS WHERE product_id IN (SELECT product_id FROM hit)
""", params=[FAILED])
impact_sql.T

The answer is right, but look at what it cost. `depth < 10` is a guess: a deeper network would silently return too few suppliers. The query returns one row of totals. To explain *why* an order is at risk, or to ask the next question (is the alternative supplier safe?), you write another long query and re-implement the same guard.

## 3. The same question in PyRel

The ontology (`supply_chain_ontology.py`) maps the tables to concepts: `Part`, `Product`, `Supplier`, `Supply`, `QualifiedSupply`, `Customer`, `Order`. Notebook 01 and the design doc explain how it's built. Here we add rules on top of it. Like the ontology, these rules were generated with generative AI and verified by running them, not written by hand.

A rule says what is true, not how to loop. "A supplier relies on another at some tier" is two lines: a direct dependency, or a dependency of something you already rely on. RelationalAI keeps deriving facts until nothing new appears. A fact that is already known is not derived again, so the cycle ends on its own, and there is no depth cap to guess.

In [ ]:
from relationalai.semantics import Integer
from supply_chain_ontology import Customer, Order, Part, Product, QualifiedSupply, Supplier, Supply, model

# How to read a rule:  model.where(<conditions>).define(<new fact>)
#   "wherever the conditions hold, this fact is true". Rules add facts to the model; nothing runs yet.
#   Queries further down trigger the evaluation, in the RelationalAI engine, not in this notebook.
# A bare concept name (Part, Supplier) is one variable throughout a rule. Concept.ref() creates an
# extra, independent variable of that type: a, b and c below are three different parts.

# Bill of materials: an assembly includes a part at some level.
# A Relationship has no key: it is just a set of (assembly, part) facts.
Part.includes = model.Relationship(f"{Part:assembly} includes {Part:part} at some level")
a, b, c, qty = Part.ref(), Part.ref(), Part.ref(), Integer.ref()
model.where(a.contains(b, qty)).define(a.includes(b))                         # base case: a direct child
model.where(a.includes(b), b.contains(c, qty)).define(a.includes(c))          # recursive case: a child of something a includes

# Supply network: a supplier relies on another at some tier. Same pattern, cycle included:
# the engine keeps applying the rules until no new fact appears, and a known fact isn't added twice.
Supplier.relies_on = model.Relationship(f"{Supplier:buyer} relies on {Supplier:upstream} at some tier")
s1, s2, s3 = Supplier.ref(), Supplier.ref(), Supplier.ref()
model.where(s1.depends_on(s2)).define(s1.relies_on(s2))                       # direct dependency
model.where(s1.relies_on(s2), s2.depends_on(s3)).define(s1.relies_on(s3))     # one tier further up

# The failed supplier is down, and so is everyone who relies on it.
# A Relationship with only a concept in it is a yes/no flag: Supplier.is_down() is true or absent.
Supplier.is_down = model.Relationship(f"{Supplier} is down")
model.where(Supplier.id == FAILED).define(Supplier.is_down())
failed = Supplier.ref()                                                       # a second supplier variable
model.where(Supplier.relies_on(failed), failed.id == FAILED).define(Supplier.is_down())

# A part has no working supplier if it has suppliers, and none of them is still up.
# model.not_(...) means "there is no such case": no Supply of this part whose supplier is not down.
Part.has_no_supplier = model.Relationship(f"{Part} has no working supplier")
any_supply, other = Supply.ref(), Supply.ref()                               # two independent Supply variables
model.where(any_supply.part == Part,
            model.not_(other.part == Part, model.not_(other.supplier.is_down()))).define(Part.has_no_supplier())

# A qualified supplier is approved for a part but doesn't deliver it yet. It is a usable alternative if
#   (1) the part has lost all its current suppliers (qualified suppliers don't count), and
#   (2) this approved, not-yet-active supplier isn't down itself, at any tier.
QualifiedSupply.is_usable = model.Relationship(f"{QualifiedSupply} is a usable alternative")
model.where(QualifiedSupply.part.has_no_supplier(),                         # (1)
            model.not_(QualifiedSupply.supplier.is_down())).define(QualifiedSupply.is_usable())   # (2)

Now ask questions against those rules. The first query starts the reasoning engine and indexes the tables, which takes a few minutes. Later queries are fast.

**Which orders are at risk?** Order → product → top part → any part it includes → that part has no supplier. One query, one row per order, with the customer and due date we need for the plan.

> **PyRel vs SQL:** the query reads like the question. Both recursions are hidden behind `includes` and `has_no_supplier`, which are defined once in the rules above. The guarded SQL returned one row of totals. Getting one row per order means rewriting it, with the guards again.

In [ ]:
# A query is  model.where(<conditions>).select(<columns>).to_df(): to_df() runs it in the engine.
# The conditions chain through the model: Order -> Product -> top part -> any included part
# that has no working supplier. includes and has_no_supplier are the rules from the cell above.
# top and part are two separate Part variables; Order, Product and Customer are one variable each.
top, part = Part.ref(), Part.ref()
at_risk = model.where(
    Order.product(Product), Product.top_part(top), top.includes(part), part.has_no_supplier(), Order.customer(Customer),
).select(                                                                     # alias() names the output columns
    Order.id.alias("order_id"), Product.id.alias("product_id"), Product.name.alias("product_name"),
    Customer.id.alias("customer_id"), Customer.region.alias("region"), Order.quantity.alias("quantity"),
    Order.due_date.alias("due_date"), Order.value.alias("value"), part.id.alias("missing_part"),
).to_df().drop_duplicates("order_id")                                        # one row per order and missing part

at_risk = at_risk.astype({"quantity": int, "value": float})
at_risk["due_date"] = pd.to_datetime(at_risk["due_date"]).dt.date
print(f"{len(at_risk)} orders at risk, worth {at_risk.value.sum():,.2f}")
assert len(at_risk) == 568 and round(at_risk.value.sum(), 2) == 6840136.49   # matches the SQL above
at_risk.head()

**Why are they at risk?** The rules are still in the model, so explaining the answer is another short query, not a new CTE: which suppliers are down, and which dependency links between them carry the failure.

> **PyRel vs SQL:** the explanation comes from the same facts that produced the answer. In SQL, the reason is in the `path` array inside the CTE. To show it, you change the CTE to return the paths and unnest them, and you need the guard again.

In [ ]:
# Three short queries over facts the rules already derived; no new recursion.
# Which suppliers are down? A flag in where() filters: only suppliers with is_down() true.
down = model.where(Supplier.is_down()).select(
    Supplier.id.alias("supplier_id"), Supplier.tier.alias("tier"), Supplier.region.alias("region"),
).to_df().sort_values("supplier_id")

# Which direct dependencies carry the failure? Pairs (Supplier, up) that are both down and linked.
up = Supplier.ref()                                                           # the upstream supplier: a second variable
chain = model.where(Supplier.is_down(), up.is_down(), Supplier.depends_on(up)).select(
    Supplier.id.alias("buyer"), up.id.alias("depends_on"),
).to_df().sort_values(["buyer", "depends_on"])

# Which parts have lost all their current suppliers?
unsourced = model.where(Part.has_no_supplier()).select(
    Part.id.alias("part_id"), Part.name.alias("name"), Part.on_hand.alias("on_hand"),
).to_df()

display(down)
display(chain)
display(unsourced)
assert down.supplier_id.tolist() == ["S-T1-02", "S-T1-05", "S-T2-03", "S-T2-08", "S-T3-07"]
assert unsourced.part_id.tolist() == ["P-141"]

**How many sensors do those orders need?** A sensor can sit several levels deep, so the quantity per unit is the product of the quantities along the path. That is the same recursive pattern with arithmetic in it.

One caveat: rules derive *sets* of facts. If a product reached the same part along two different paths with the same quantity, the two facts would be identical and count once. In this data every product reaches `P-141` once, through the control board `P-040`. For a BOM with repeated paths you would add the intermediate part to the relationship to keep paths apart.

**Is the qualified alternative safe?** `QualifiedSupply.is_usable` asks whether the alternative is itself hit by the failure, at any tier. That is the question a planner asks next, and in SQL it is another guarded recursion.

> **PyRel vs SQL:** each new question reuses what is already derived. `needs` is the same two-line recursive pattern with a multiplication in it. `is_usable` doesn't recurse at all: it reuses `is_down`. In SQL, the quantity calculation is a third recursive CTE that carries a running product along the path. The safety check is a fourth, or a copy of the first, and each one needs the same cycle guard and depth cap.

In [ ]:
# Quantity of a part per unit of an assembly, multiplied along the BOM path.
# Same recursive pattern as includes, but each fact also carries a number:
# q1 and q2 are Integer variables that bind to the quantities, and q1 * q2 is computed in the engine.
Part.needs = model.Relationship(f"{Part:assembly} needs {Part:part} {Integer:qty} times per unit")
p1, p2, p3, q1, q2 = Part.ref(), Part.ref(), Part.ref(), Integer.ref(), Integer.ref()
model.where(p1.contains(p2, q1)).define(p1.needs(p2, q1))                                    # direct child: its own quantity
model.where(p1.needs(p2, q1), p2.contains(p3, q2)).define(p1.needs(p3, q1 * q2))           # one level deeper: multiply

# Per order: how many of the missing part one unit of its product needs. per_unit binds to the quantity in needs.
top, part, per_unit = Part.ref(), Part.ref(), Integer.ref()
sensors = model.where(Order.product(Product), Product.top_part(top), top.needs(part, per_unit),
                      part.has_no_supplier()).select(
    Order.id.alias("order_id"), per_unit.alias("per_unit")).to_df()
sensors = sensors.astype({"per_unit": int}).groupby("order_id", as_index=False).per_unit.sum()
at_risk = at_risk.drop(columns="sensors", errors="ignore").merge(sensors, on="order_id")
at_risk["sensors"] = at_risk.quantity * at_risk.per_unit

# Usable alternatives: the is_usable rule does the work. A dotted path such as QualifiedSupply.supplier.id
# follows a key field to the Supplier and reads its id.
options = model.where(QualifiedSupply.is_usable()).select(
    QualifiedSupply.supplier.id.alias("supplier_id"), QualifiedSupply.supplier.region.alias("region"),
    QualifiedSupply.part.id.alias("part_id"), QualifiedSupply.capacity_per_week.alias("capacity_per_week"),
    QualifiedSupply.lead_time_days.alias("lead_time_days"), QualifiedSupply.unit_price.alias("unit_price"),
).to_df()

print(f"sensors needed: {at_risk.sensors.sum():,}")
display(options)
assert at_risk.sensors.sum() == 29286 and options.supplier_id.tolist() == ["S-T1-01"]

### Where PyRel does better than SQL here, and why

| Question | SQL | PyRel | Why the difference |
|---|---|---|---|
| Which part has two suppliers? (section 1) | One join | — | SQL is the right tool for a flat lookup. |
| Who is down, at any tier? | Recursive CTE with a visited-path array and `depth < 10` | `relies_on`: two lines | Rules stop when no new fact appears, so the cycle ends on its own and there's no depth to guess. |
| Which orders are at risk? | A second recursion for the BOM, plus set logic | One query over `includes` and `has_no_supplier` | The recursions are defined once, as named facts, and queries just use them. |
| Why is each order at risk? | Rewrite the CTE to return and unnest the paths | One query over `is_down` and `depends_on` | The facts behind the answer stay in the model and can be queried. |
| How many sensors are needed? | A third recursion carrying a running product | `needs`: the same pattern, with `q1 * q2` | Arithmetic fits into the same recursive rule. |
| Is the alternative safe? | Another guarded recursion | `is_usable` reuses `is_down` | Each new question builds on facts that are already derived. |

SQL can answer every one of these questions. PyRel wins on what each next question costs: in SQL, every follow-up is a new recursive query with its own guard; in PyRel, it's a short query or rule on top of what's already there.

## 4. Build the facts pack

The alternative needs 70 days to start, so it delivers from 2027-01-04. Orders due before then can only be served from the sensors in stock. That gap is the heart of the plan.

We compute every number here, in Python, from what RelationalAI derived: totals, the gap, the shortfall. The LLM gets the result as a small JSON document. An LLM is good at writing and structuring; it is not a calculator, and it should not be the place where numbers are made.

In [ ]:
alt = options.iloc[0]
sensor = unsourced.iloc[0]
current = sql("SELECT supplier_id, unit_price FROM SUPPLIER_PARTS WHERE part_id = ?", params=[sensor.part_id])

first_delivery = AS_OF + dt.timedelta(days=int(alt.lead_time_days))
before = at_risk[at_risk.due_date < first_delivery]
stock = int(sensor.on_hand)

facts = {
    "as_of": str(AS_OF),
    "event": {"failed_supplier": FAILED, "tier": int(down.set_index("supplier_id").tier[FAILED])},
    "dependency_chain": [f"{r.buyer} depends on {r.depends_on}" for r in chain.itertuples()],
    "suppliers_down": down.supplier_id.tolist(),
    "part_without_supplier": {
        "part_id": sensor.part_id, "name": sensor["name"], "on_hand": stock,
        "current_suppliers_all_down": current.supplier_id.tolist(),
        "highest_current_unit_price": float(current.unit_price.max()),
    },
    "impact": {
        "orders": len(at_risk), "order_value": round(at_risk.value.sum(), 2),
        "sensors_needed": int(at_risk.sensors.sum()),
        "products": sorted(at_risk.product_name.unique()),
        "by_region": {r: {"orders": len(g), "value": round(g.value.sum(), 2)} for r, g in at_risk.groupby("region")},
    },
    "until_alternative_delivers": {
        "orders_due": len(before), "order_value": round(before.value.sum(), 2),
        "sensors_needed": int(before.sensors.sum()),
        "sensor_shortfall_after_stock": int(before.sensors.sum()) - stock,
    },
    "qualified_alternative": {
        "supplier_id": alt.supplier_id, "region": alt.region, "capacity_per_week": int(alt.capacity_per_week),
        "lead_time_days": int(alt.lead_time_days), "unit_price": float(alt.unit_price),
        "first_delivery": str(first_delivery), "relies_on_failed_supplier": False,
    },
}
print(json.dumps(facts, indent=2))

## 5. Ask Cortex for a mitigation plan

`AI_COMPLETE` runs inside Snowflake; the facts don't leave the platform. The prompt sets the rules: use only the facts, copy numbers exactly, list unknowns as open questions. `temperature` 0 keeps the output stable between runs. The prompt is passed as a bind parameter, not pasted into the SQL text.

This is one call with about 1,000 input tokens, so the cost is small. The model writes up to `max_tokens` output tokens on top of that.

In [ ]:
from IPython.display import Markdown

PROMPT = """You are a supply chain planner. Write a mitigation plan for the disruption described in FACTS.

Rules:
- Use only FACTS. Do not invent suppliers, dates, quantities or prices.
- When you use a number, copy it exactly from FACTS. Do not calculate new numbers.
- If the plan needs something FACTS doesn't contain, list it under "Open questions".

Structure, in markdown, under 350 words:
1. Situation (three sentences, including why a two-supplier part still failed)
2. Immediate actions (this week)
3. Bridge until the alternative supplier delivers
4. Structural fixes
5. Open questions

FACTS:
"""

prompt = PROMPT + json.dumps(facts, indent=2)
plan = sql("""
SELECT AI_COMPLETE(
    model => ?,
    prompt => ?,
    model_parameters => {'temperature': 0, 'max_tokens': 1500}
) AS plan
""", params=[LLM, prompt]).plan[0]
plan = json.loads(plan) if plan.startswith('"') else plan   # the string result comes back JSON-quoted
display(Markdown(plan))

## 6. Check the plan against the facts

A fluent plan can still contain a number that came from nowhere. The check below collects every number in the plan and looks it up in the facts pack. When we tested an earlier prompt, the model subtracted stock from demand on its own. The arithmetic was right, but nobody had checked it. That is why the shortfall is now in the facts.

This check catches invented or recalculated numbers. It does not judge whether the advice is sound; a planner still reads the plan.

In [ ]:
def numbers(text):
    """Every number in a text, as floats: '6,840,136.49' -> 6840136.49, '2027-01-04' -> 2027, 1, 4."""
    return {float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}


allowed = numbers(json.dumps(facts)) | numbers(PROMPT)
ungrounded = sorted(n for n in numbers(plan) if n not in allowed and n > 10)   # skip list numbering
if ungrounded:
    print("Numbers in the plan that are not in the facts:", ungrounded)
else:
    print(f"All {len(numbers(plan))} numbers in the plan appear in the facts pack.")

## 7. Keep the result

The plan and the at-risk orders become ordinary tables, so a dashboard, a task or a colleague can pick them up.

In [ ]:
orders_out = at_risk[["order_id", "product_id", "customer_id", "region", "quantity", "due_date", "value", "sensors"]].copy()
orders_out["due_before_alternative"] = orders_out.due_date < first_delivery
orders_out.columns = [c.upper() for c in orders_out.columns]
session.write_pandas(orders_out, "ORDERS_AT_RISK", auto_create_table=True, overwrite=True)

plan_out = pd.DataFrame([{"AS_OF": AS_OF, "FAILED_SUPPLIER": FAILED, "MODEL": LLM,
                          "FACTS": json.dumps(facts), "PLAN": plan, "UNGROUNDED_NUMBERS": json.dumps(ungrounded)}])
session.write_pandas(plan_out, "MITIGATION_PLANS", auto_create_table=True, overwrite=True)
sql("SELECT COUNT(*) AS orders_at_risk FROM ORDERS_AT_RISK")

## What you've seen

| Question | SQL | PyRel |
|---|---|---|
| Who relies on `S-T3-07`, at any tier? | Recursive CTE with a path array and a guessed depth cap; unguarded, it can run forever on the cycle | Two rules; the cycle ends by itself |
| Which parts lost every supplier? | `GROUP BY ... HAVING COUNT_IF(...) = 0` on top of the recursion | One rule with a double negation, outside the recursion |
| Which orders are hit? | A second recursion over the BOM, joined to the first | One query that reuses `includes` |
| Why, and is the alternative safe? | New queries, same guard again | Short queries on rules already in the model |

**Python libraries are a valid alternative.** NetworkX handles the cycle as well. You then pull the tables into memory, keep two graphs in sync by hand, and own the scaling yourself. RelationalAI keeps the reasoning next to the data in Snowflake, as reusable rules. It has limits too: engines take a few minutes to start, and the model has to fit the engine's memory.

**The LLM's role is narrow on purpose.** It turns facts into a plan a person can act on. It does not query data and it does not calculate. The grounding check makes that rule visible.

### Try next

- Change `FAILED` to another supplier, for example `S-T2-14` on the cycle, and rerun from section 3. The `assert` lines check the planted case, so remove them first.
- Add a second row to `QUALIFIED_SUPPLIERS` that *does* rely on `S-T3-07`, and watch `is_usable` reject it.
- Ask for the plan in a different format: a customer email per region, or a one-paragraph summary for management.